# Hebrew Acronym Disambiguation: Evaluation Protocol

This computational appendix evaluates Hebrew acronym disambiguation under two task formulations: **free generation**, without candidate expansions or reference answers, and **candidate selection**, with a fixed candidate inventory and recorded candidate order. Each language model is evaluated on the same items in both tasks.

The technical pilot uses the first **10 of 289 development items** in `data/splits/dev_items.csv`. The test set contains **381 items**: 253 from Knesset text, 41 from Wikipedia, and 87 AI-authored items. It is therefore not a natural-text-only benchmark. Source provenance is documented in `data/README.md` and `data/mined/DATASET_CARD.md` in the source archive. The separate 62-item encoder development cohort is not the pilot cohort used here.

The protocol separates environment preparation, technical pilot inspection, and test evaluation. Software checks establish implementation properties; they do not establish live service availability, successful collection, or model performance. Results are supported by the identified run artifacts produced during execution. The operational reference is `docs/reproducible_evaluation.md` in the source archive.


## 1. Source and Environment Reproducibility

`CODE_REVISION` identifies an exact 40-character Git commit. In archive mode, upload `evaluation-release.tar` through the Colab Files panel and set `ARCHIVE_PATH`; `ARCHIVE_SHA256` must match the independently supplied release checksum. The installation checks both the archive checksum and its embedded commit before installing the package. Archive mode does not require publishing the commit. The alternative Git mode requires that the specified commit be accessible remotely.

The installation verifies the project's pinned dependencies and reports conflicts among other preinstalled Colab packages separately. Restart the runtime and repeat setup after changing the source revision or when Colab requests a restart for dependencies already loaded in memory. Subsequent cells use the installed package.


In [1]:
import hashlib, importlib.metadata, json, os, re, shutil, subprocess, sys, tarfile, tempfile, time, urllib.request
from pathlib import Path
CODE_REVISION = "79b767a3f04bfadc9536d505b11f73ac70a99b5c"
SOURCE_MODE = "archive"
ARCHIVE_PATH = Path("/content/evaluation-release.tar")
ARCHIVE_SHA256 = "1a038ba220331bf2f5dc7271f3a01e898faf52a2c1dac0685ab87c478f867efc"
REPOSITORY_URL = "https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git"
# Retrieve the identified public commit and recreate the verified release archive.
# This avoids a second manual upload; the independent archive hash is still enforced.
if not ARCHIVE_PATH.exists():
    archive_checkout = Path(tempfile.mkdtemp(prefix="gemini381-archive-"))
    subprocess.run(["git", "init", str(archive_checkout)], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(archive_checkout), "fetch", "--depth", "1", REPOSITORY_URL, CODE_REVISION], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(archive_checkout), "archive", "--format=tar", "--output=" + str(ARCHIVE_PATH), CODE_REVISION], check=True)
archive_bytes = ARCHIVE_PATH.read_bytes()
assert hashlib.sha256(archive_bytes).hexdigest() == ARCHIVE_SHA256, "Release archive mismatch"
assert subprocess.check_output(["git", "get-tar-commit-id"], input=archive_bytes).decode().strip() == CODE_REVISION
REPO_DIR = Path("/content/hebrew-eval-" + CODE_REVISION)
REPO_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(ARCHIVE_PATH, "r:") as archive:
    archive.extractall(REPO_DIR, filter="data")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    requirement = line.strip()
    if requirement and not requirement.startswith("#"):
        package_name, required_version = requirement.split("==", 1)
        assert importlib.metadata.version(package_name) == required_version, package_name
pip_report = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if pip_report.returncode:
    print("Colab environment dependency report (separate from verified project pins):")
    print(pip_report.stdout or pip_report.stderr)
print("Installed source commit:", CODE_REVISION)
print("Verified archive SHA-256:", ARCHIVE_SHA256)

Colab environment dependency report (separate from verified project pins):
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.32.5.
numba 0.61.2 has requirement numpy<2.3,>=1.24, but you have numpy 2.5.3.
torchvision 0.26.0+cpu has requirement torch==2.11.0, but you have torch 2.8.0.
diffusers 0.40.0 has requirement huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2.
gradio 6.29.0 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.

Installed source commit: 79b767a3f04bfadc9536d505b11f73ac70a99b5c
Verified archive SHA-256: 1a038ba220331bf2f5dc7271f3a01e898faf52a2c1dac0685ab87c478f867efc


## 2. Persistent Storage, Credentials, and Expenditure

`RUN_NAME` identifies the persistent Drive directory. Retain this name and the same settings when resuming after a disconnection or selecting additional systems. A changed code revision or configuration requires a new identified session; preserve previous directories and carry their expenditure into the new session exactly once.

Set `PRIOR_SPEND_ILS` to earlier API expenditure outside this session, including failed requests and conservative allowances for uncertain charges. Its initial value, `None`, requires explicit reconciliation before execution. The combined expenditure ceiling is **100 ILS** across systems, pilots, test requests, and retries. The conversion allowance of 4 ILS/USD includes a fee and tax margin; it is not a quoted exchange rate. Changing system selection does not reset expenditure.

`SELECTED_SYSTEMS` determines which of the six systems are attempted. Store `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, and `XAI_API_KEY` in **Colab Secrets** and enable access for this notebook. Credentials must not appear in notebook cells or outputs. Missing credentials are reported per system and do not prevent other systems from proceeding. Use one writer per run directory; mounted Drive storage does not guarantee durability during a storage failure. Paid runtime purchases and automatic credit replenishment are outside this protocol.


In [3]:
from google.colab import drive, userdata
import importlib.metadata
import platform
import torch
from hebrew_acronyms.staged_evaluation import (
    SYSTEM_NAMES, prepare_session, run_system_pilot, review_system_pilot,
    run_system_full, session_summary,
)
from hebrew_acronyms.test_evaluation import summarize_evaluation
from hebrew_acronyms.models.qwen.eval import inspect_ollama
from hebrew_acronyms.models.openai.eval import inspect_openai_model
from hebrew_acronyms.models.anthropic.eval import inspect_anthropic_model
from hebrew_acronyms.models.xai.eval import inspect_xai_model
from hebrew_acronyms.models.common.http import request_json

RUN_NAME = "evaluation-gemini-381-20261010-79b767a"
SELECTED_SYSTEMS = ["gemini"]
PRIOR_SPEND_ILS = 21.6620734  # Required: reconciled prior expenditure and uncertain-cost allowances.
ILS_PER_USD_ALLOWANCE = 4.0  # Conversion allowance including fees and taxes; not a quoted exchange rate.
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple stable run name")
if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
    raise ValueError("Select one or more of the approved systems, without duplicates")
if PRIOR_SPEND_ILS is None:
    raise ValueError("Account for earlier attempts in PRIOR_SPEND_ILS before proceeding")
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/NLP/evaluation-runs") / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)  # First initialization only; never overwrite an existing session.
probe = OUTPUT_ROOT / f"drive-access-{time.time_ns()}.txt"
with probe.open("x") as stream:
    stream.write(RUN_NAME)
if probe.read_text() != RUN_NAME:
    raise RuntimeError("Drive read-after-write check failed")
AVAILABILITY = {name: {"status": "not_selected"} for name in SYSTEM_NAMES}
SECRET_NAMES = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "xai": "XAI_API_KEY"}
for name in SELECTED_SYSTEMS:
    if name not in SECRET_NAMES:
        continue
    secret_name = SECRET_NAMES[name]
    os.environ.pop(secret_name, None)  # Remove stale credentials before reading this notebook's Secrets.
    try:
        secret_value = userdata.get(secret_name)
        if not secret_value:
            raise ValueError("empty")
        os.environ[secret_name] = secret_value
        AVAILABILITY[name] = {"status": "credentials_loaded"}
        del secret_value
    except Exception:
        AVAILABILITY[name] = {"status": "unavailable", "reason": "missing_secret"}
ENVIRONMENT = {
    "code_revision": CODE_REVISION, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "requests", "numpy")},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "archive_sha256": ARCHIVE_SHA256 if SOURCE_MODE == "archive" else None,
}
with (OUTPUT_ROOT / f"environment-{time.time_ns()}.json").open("x") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)
print(ENVIRONMENT)
print("Persistent outputs:", OUTPUT_ROOT)
print(AVAILABILITY)

Mounted at /content/drive
{'code_revision': '79b767a3f04bfadc9536d505b11f73ac70a99b5c', 'python': '3.13.16', 'packages': {'torch': '2.8.0', 'transformers': '4.57.6', 'requests': '2.32.5', 'numpy': '2.5.3'}, 'gpu': None, 'archive_sha256': '1a038ba220331bf2f5dc7271f3a01e898faf52a2c1dac0685ab87c478f867efc'}
Persistent outputs: /content/drive/MyDrive/NLP/evaluation-runs/evaluation-gemini-381-20261010-79b767a
{'qwen': {'status': 'not_selected'}, 'gemini': {'status': 'credentials_loaded'}, 'openai': {'status': 'not_selected'}, 'anthropic': {'status': 'not_selected'}, 'qwen14': {'status': 'not_selected'}, 'xai': {'status': 'not_selected'}, 'dictalm': {'status': 'not_selected'}}


## 3. Local Model Configuration

Selecting `qwen` downloads **Qwen2.5 7B** (`qwen2.5:7b`); selecting `qwen14` downloads **Qwen2.5 14B** (`qwen2.5:14b`, Q4_K_M). Ollama is installed only when either is selected. Use a free GPU when available. Before the pilot, inspect the resolved model digest, server version, template, quantization, stop sequences, and generation settings. These properties identify the local model used in the run; the tag alone is insufficient.

Generation uses temperature 0, seed 42, and a 512-token output limit. Hardware and implementation differences may still affect reproducibility. A setup failure is recorded without substituting another model, and other available systems may proceed.

Qwen14 is preloaded with an empty Ollama request before evaluation. Preparation time is recorded separately; the 120-second inference timeout and generation settings remain unchanged.


In [4]:
QWEN_MODEL = "qwen2.5:7b"
QWEN14_MODEL = "qwen2.5:14b"
QWEN_OPTIONS = {"temperature": 0, "seed": 42, "num_predict": 512}
OLLAMA_URL = "http://localhost:11434"
QWEN_IDENTITIES = {}
QWEN_STABLE_FIELDS = ("model", "digest", "server_version", "options",
                      "template", "model_parameters", "details", "model_info")
import requests
for qwen_name, qwen_model in (("qwen", QWEN_MODEL), ("qwen14", QWEN14_MODEL)):
    if qwen_name not in SELECTED_SYSTEMS:
        continue
    try:
        if shutil.which("ollama") is None:
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
            installer = Path("/content/ollama-install.sh")
            urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
        import requests
        try:
            requests.get(OLLAMA_URL + "/api/version", timeout=5).raise_for_status()
        except requests.RequestException:
            ollama_log = (OUTPUT_ROOT / f"ollama-{time.time_ns()}.log").open("w")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
            for _ in range(30):
                time.sleep(1)
                try:
                    requests.get(OLLAMA_URL + "/api/version", timeout=2).raise_for_status()
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect its saved log")
        subprocess.run(["ollama", "pull", qwen_model], check=True)
        inspection = inspect_ollama(qwen_model, options=QWEN_OPTIONS)
        if qwen_name == "qwen14" and inspection.get("details", {}).get("quantization_level") != "Q4_K_M":
            raise ValueError("Qwen14 quantization differs from the specified Q4_K_M")
        QWEN_IDENTITIES[qwen_name] = inspection
        # Download timestamps are session evidence, separate from stable model identity.
        with (OUTPUT_ROOT / f"{qwen_name}-inspection-{time.time_ns()}.json").open("x") as stream:
            json.dump(inspection, stream, ensure_ascii=False, indent=2)
        # Load Qwen14 before timing any evaluation request; no prompt is supplied.
        if qwen_name == "qwen14":
            preload_started = time.perf_counter()
            preload_response = requests.post(
                OLLAMA_URL + "/api/generate",
                json={"model": qwen_model, "stream": False, "keep_alive": "30m",
                      "options": QWEN_OPTIONS}, timeout=300)
            preload_response.raise_for_status()
            preload = preload_response.json()
            if preload.get("done") is not True or preload.get("response", "") or preload.get("eval_count", 0):
                raise RuntimeError("Expected an empty, completed Qwen14 preload")
            preload_evidence = {"model": qwen_model, "digest": inspection["digest"],
                                "elapsed_seconds": time.perf_counter() - preload_started,
                                "request_kind": "empty_preload", "result": preload}
            with (OUTPUT_ROOT / f"{qwen_name}-preload-{time.time_ns()}.json").open("x") as stream:
                json.dump(preload_evidence, stream, ensure_ascii=False, indent=2)
        frozen_identity = {key: inspection[key] for key in QWEN_STABLE_FIELDS}
        AVAILABILITY[qwen_name] = {"status": "available"}
        print(json.dumps(frozen_identity, ensure_ascii=False, indent=2))
    except Exception:
        AVAILABILITY[qwen_name] = {"status": "unavailable", "reason": "local_model_setup_failed"}
print({name: AVAILABILITY[name] for name in ("qwen", "qwen14")})

{'qwen': {'status': 'not_selected'}, 'qwen14': {'status': 'not_selected'}}


## 4. API Model Availability and Cost Assumptions

The API systems are **Gemini** (`gemini-3.8-flash`), **GPT-4.1 mini** (`gpt-4.1-mini-2025-04-14`), and **Claude Haiku 5.5** (`claude-haiku-5-5`). Availability lookups do not generate text and do not establish sufficient credit or quota. The report distinguishes unavailable and unselected systems; only available selected systems enter collection.

Gemini uses low thinking, and Claude uses adaptive thinking with low effort. Output limits include provider-billed thinking where applicable. Exact settings are recorded with the run, and returned model identities are inspected in the pilot.

The rates below are expressed in USD per million tokens. Verify them against the official [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing), [GPT-4.1 mini specification](https://developers.openai.com/api/docs/models/gpt-4.1-mini), and [Claude Haiku 5.5 specification](https://platform.claude.com/docs/en/models/haiku-5-5/overview) before collection. Cached input is conservatively charged at the full input rate. Token-based expenditure is an estimate from reported usage, not a settled provider invoice.

Grok 4.7 uses low reasoning effort, no tools, and a 1,024-token output cap.
Its input/output rates are USD 2/6 per million tokens; cached input is conservatively charged at the full input rate.
Qwen2.5 14B uses the same sampling settings as 7B; its resolved digest and quantization are recorded separately.
These additions require their own inspected technical pilots. Earlier collection packages and results remain separate.
Sources: [xAI model specification](https://docs.x.ai/developers/models/grok-4.7),
[Qwen2.5 model library](https://ollama.com/library/qwen2.5:14b).


In [6]:
SYSTEMS = [
    {"name": "qwen", "provider": "qwen", "model": QWEN_MODEL,
     "settings": {"expected_digest": QWEN_IDENTITIES.get("qwen", {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "gemini", "provider": "gemini", "model": "gemini-3.8-flash",
     "settings": {"timeout": 120, "generation_config": {
         "maxOutputTokens": 1024,
         "thinkingConfig": {"thinkingLevel": "low"}}}},
    {"name": "openai", "provider": "openai", "model": "gpt-4.1-mini-2025-04-14",
     "settings": {"timeout": 120, "max_output_tokens": 512, "temperature": 0}},
    {"name": "anthropic", "provider": "anthropic", "model": "claude-haiku-5-5",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
    {"name": "qwen14", "provider": "qwen", "model": QWEN14_MODEL,
     "settings": {"expected_digest": QWEN_IDENTITIES.get("qwen14", {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "xai", "provider": "xai", "model": "grok-4.7",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
]
RATES = {"qwen": {"input": 0.0, "output": 0.0},
         "gemini": {"input": 0.75, "output": 3.75},
         "openai": {"input": 0.40, "output": 1.60},
         "anthropic": {"input": 0.10, "output": 0.50},
         "qwen14": {"input": 0.0, "output": 0.0},
         "dictalm": {"input": 0.0, "output": 0.0},
         "xai": {"input": 2.0, "output": 6.0}}
for system in SYSTEMS:
    name = system["name"]
    if name not in SELECTED_SYSTEMS or name in {"qwen", "qwen14"} or AVAILABILITY[name]["status"] == "unavailable":
        continue
    if name == "openai":
        info = inspect_openai_model(model=system["model"])
    elif name == "anthropic":
        info = inspect_anthropic_model(model=system["model"])
    elif name == "xai":
        info = inspect_xai_model(model=system["model"])
    else:
        payload, info = request_json(
            requests.get, "https://generativelanguage.googleapis.com/v1beta/models/" + system["model"],
            provider="Gemini", secret=os.environ["GEMINI_API_KEY"], env_name="GEMINI_API_KEY",
            headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
        if payload is not None and payload.get("name") == "models/" + system["model"] and "generateContent" in (payload.get("supportedGenerationMethods") or []):
            info["status"] = "available"
    # Save only allowlisted diagnostics, never upstream body/message or credentials.
    AVAILABILITY[name] = {key: info.get(key) for key in
        ("status", "http_status", "error_category", "provider_blocked", "retry_after")}
def available_selected_systems():
    # Recheck current selection and availability before collection.
    if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
        raise ValueError("Select approved systems without duplicates")
    return [s for s in SYSTEMS if s["name"] in SELECTED_SYSTEMS
            and AVAILABILITY.get(s["name"], {}).get("status") == "available"]
with (OUTPUT_ROOT / f"availability-{time.time_ns()}.json").open("x") as stream:
    json.dump(AVAILABILITY, stream, indent=2)
print(json.dumps(AVAILABILITY, indent=2))
DEV_PATH = REPO_DIR / "data/splits/dev_items.csv"
TEST_PATH = REPO_DIR / "data/splits/test_items.csv"
def model_identity(system):
    return ({key: QWEN_IDENTITIES[system["name"]][key] for key in QWEN_STABLE_FIELDS}
            if system["provider"] == "qwen" else None)

{
  "qwen": {
    "status": "not_selected"
  },
  "gemini": {
    "status": "available",
    "http_status": 200,
    "error_category": null,
    "provider_blocked": false,
    "retry_after": null
  },
  "openai": {
    "status": "not_selected"
  },
  "anthropic": {
    "status": "not_selected"
  },
  "qwen14": {
    "status": "not_selected"
  },
  "xai": {
    "status": "not_selected"
  },
  "dictalm": {
    "status": "not_selected"
  }
}


## 5. Technical Development Pilot

Each system receives the same first ten development items in both tasks: **20 logical responses per system**. Candidate order is derived from the item identity and a fixed candidate-order seed of 42. Each request is journalled before execution and its outcome after return.

At most two attempts are allowed per logical request. Explicit provider blocks stop collection for that provider, and transient retries obey bounded waiting. Interrupted requests with uncertain completion are retained for inspection and are not automatically resent. Their possible expenditure remains included through conservative allowances.

Pilot and test journals for every system contribute to the shared expenditure ceiling, including systems not currently selected. Preserve all journals when resuming. An incomplete pilot does not qualify its system for test evaluation.


In [7]:
# Offline identity checks before any generation request.
from hebrew_acronyms.test_evaluation import prepare_evaluation
from hebrew_acronyms.models.common.pairs import load_rows
GEMINI_SYSTEM = next(s for s in SYSTEMS if s['name'] == 'gemini')
assert len(load_rows(TEST_PATH)) == 381
assert len(load_rows(DEV_PATH)) == 289
comparison = {}
for cohort, path, old_relative in (
    ('full_test', TEST_PATH, 'saved-results/colab-runs/evaluation-20261009-a319f9d/extracted/openai/full-test/manifest.json'),
    ('dev_pilot', DEV_PATH, 'saved-results/colab-runs/evaluation-gemini-retry-20261009-a319f9d/extracted/gemini/dev-pilot/manifest.json')):
    previous = json.loads((REPO_DIR / old_relative).read_text())['identity']
    derived = prepare_evaluation(path, Path(tempfile.mkdtemp()) / cohort,
        cohort=cohort, systems=[GEMINI_SYSTEM], code_revision=CODE_REVISION)['identity']
    assert derived['protocol'] == previous['protocol'] and derived['seed'] == previous['seed']
    old_rows = {r['item_id']:r for r in previous['rows']}
    assert all(r == old_rows[r['item_id']] for r in derived['rows'])
    old_requests = {(r['item_id'],r['task']):r for r in previous['requests']}
    for request in derived['requests']:
        previous_request = old_requests[(request['item_id'],request['task'])]
        assert all(request[k] == previous_request[k] for k in ('prompt','prompt_sha256','shown_order'))
    comparison[cohort] = {'items':len(derived['rows']),'requests':len(derived['requests']),
        'protocol':derived['protocol'],'source_sha256':derived['source_sha256'],
        'matched_reference':old_relative,'all_rows_prompts_and_candidate_orders_equal':True}
print(json.dumps(comparison, indent=2))
if 'OUTPUT_ROOT' in globals():
    with (OUTPUT_ROOT / 'input-comparison.json').open('x') as stream: json.dump(comparison, stream, indent=2)


{
  "full_test": {
    "items": 381,
    "requests": 762,
    "protocol": "fixed-cohort-item-shuffle-labels-v1",
    "source_sha256": "7d85400fe16920b9508f7fe4620fc0c651872ad6241d06b8ac8a4d932b36501d",
    "matched_reference": "saved-results/colab-runs/evaluation-20261009-a319f9d/extracted/openai/full-test/manifest.json",
    "all_rows_prompts_and_candidate_orders_equal": true
  },
  "dev_pilot": {
    "items": 10,
    "requests": 20,
    "protocol": "fixed-cohort-item-shuffle-labels-v1",
    "source_sha256": "5da0e0640bc8e841f6d4008243261d074f075feb7316414a579c1f30324665ff",
    "matched_reference": "saved-results/colab-runs/evaluation-gemini-retry-20261009-a319f9d/extracted/gemini/dev-pilot/manifest.json",
    "all_rows_prompts_and_candidate_orders_equal": true
  }
}


In [10]:
# Fixed dev pilot; continue only in this same journal.
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.common.eval import build_generate_prompt, build_select_prompt
from datetime import datetime, timezone
all_rows = load_rows(DEV_PATH) + load_rows(TEST_PATH)
max_prompt_bytes = max(len(p.encode('utf-8')) for row in all_rows for p in (
    build_generate_prompt(row['acronym'], row['sentence']),
    build_select_prompt(row['acronym'], row['sentence'], row['candidates'].split('|'))))
input_token_allowance = max_prompt_bytes + 1024
assert input_token_allowance < 100_000
OUTPUT_CAPS = {'qwen':512,'gemini':1024,'openai':512,'anthropic':1024,'qwen14':512,'xai':1024,'dictalm':512}
RESERVE_PER_CALL_USD = {name:(input_token_allowance*rate['input']+OUTPUT_CAPS[name]*rate['output'])/1_000_000 for name,rate in RATES.items()}
def current_session():
    return prepare_session(OUTPUT_ROOT,DEV_PATH,TEST_PATH,code_revision=CODE_REVISION,
        prior_spend_ils=PRIOR_SPEND_ILS,rates=RATES,reserves=RESERVE_PER_CALL_USD,ils_per_usd=ILS_PER_USD_ALLOWANCE)
def saved_events(directory):
    return [json.loads(line) for path in sorted(directory.glob('attempts*.jsonl')) for line in path.read_text().splitlines()]
assert SELECTED_SYSTEMS == ['gemini'] and AVAILABILITY['gemini']['status'] == 'available'
current_session()
PILOT_MAX_NEW_CALLS = 16
pilot_dir = OUTPUT_ROOT / 'gemini/dev-pilot'
for _ in range(PILOT_MAX_NEW_CALLS):
    events = saved_events(pilot_dir)
    if events:
        began = min(datetime.fromisoformat(e['at_utc']) for e in events if e['event']=='started')
        if (datetime.now(timezone.utc)-began).total_seconds() >= 600:
            print('Pilot paused at the 10-minute boundary between requests.'); break
    result = run_system_pilot(OUTPUT_ROOT,GEMINI_SYSTEM,max_new_calls=1)
    summary = result['summary']
    print({k:summary[k] for k in ('n_calls','n_completed','n_pending','n_ambiguous')},flush=True)
    if summary['n_completed']==20 or summary.get('provider_states') or summary['n_ambiguous']:
        break
print(json.dumps(session_summary(OUTPUT_ROOT),indent=2))

{'n_calls': 5, 'n_completed': 5, 'n_pending': 15, 'n_ambiguous': 0}
{'n_calls': 6, 'n_completed': 6, 'n_pending': 14, 'n_ambiguous': 0}
{'n_calls': 7, 'n_completed': 7, 'n_pending': 13, 'n_ambiguous': 0}
{'n_calls': 8, 'n_completed': 8, 'n_pending': 12, 'n_ambiguous': 0}
{'n_calls': 9, 'n_completed': 9, 'n_pending': 11, 'n_ambiguous': 0}
{'n_calls': 10, 'n_completed': 10, 'n_pending': 10, 'n_ambiguous': 0}
{'n_calls': 11, 'n_completed': 11, 'n_pending': 9, 'n_ambiguous': 0}
{'n_calls': 12, 'n_completed': 12, 'n_pending': 8, 'n_ambiguous': 0}
{'n_calls': 13, 'n_completed': 13, 'n_pending': 7, 'n_ambiguous': 0}
{'n_calls': 14, 'n_completed': 14, 'n_pending': 6, 'n_ambiguous': 0}
{'n_calls': 15, 'n_completed': 15, 'n_pending': 5, 'n_ambiguous': 0}
{'n_calls': 16, 'n_completed': 16, 'n_pending': 4, 'n_ambiguous': 0}
{'n_calls': 17, 'n_completed': 17, 'n_pending': 3, 'n_ambiguous': 0}
{'n_calls': 18, 'n_completed': 18, 'n_pending': 2, 'n_ambiguous': 0}
{'n_calls': 19, 'n_completed': 19, 'n_

## 6. Pilot Inspection and Test Projection

Inspect all 20 responses for coverage, model identity, finish reason, truncation, token usage, elapsed time, and estimated cost. Pilot acceptance is technical: semantic errors remain observations and do not justify repeating requests or selecting a more favorable run.

For a complete pilot, the cost projection scales from 10 to 381 items with a 25% margin and includes accumulated expenditure across systems and allowances for uncertain usage. The elapsed-time projection is descriptive rather than a completion guarantee.

After inspecting a successful pilot's records, enter its printed `pilot_identity` in `INSPECTED_PILOTS` in the next cell. The initially empty mapping records no completed inspection. Incomplete or blocked systems remain reported, while independently qualified systems may proceed. Execute these stages sequentially; **Run all** is not a substitute for pilot inspection.


In [11]:
# Inspect every pilot response before authorizing test collection.
pilot_snapshot = summarize_evaluation(OUTPUT_ROOT / 'gemini/dev-pilot')
for record in pilot_snapshot['records']:
    if record.get('status') != 'pending': print(json.dumps(record, ensure_ascii=False))
if pilot_snapshot['n_completed'] == 20:
    current_session()
    review = review_system_pilot(OUTPUT_ROOT, GEMINI_SYSTEM)
    pilot_summary = review['summary']
    for record in pilot_summary['records']:
        print(json.dumps(record, ensure_ascii=False))
    assert pilot_summary['n_completed'] == 20 and pilot_summary['n_ambiguous'] == 0
    assert all(r['response_metadata'].get('model_version') == 'gemini-3.8-flash' and r['response_metadata'].get('finish_reason') == 'STOP' for r in pilot_summary['records'])
    pilot_events = saved_events(OUTPUT_ROOT / 'gemini/dev-pilot')
    pilot_start = min(datetime.fromisoformat(e['at_utc']) for e in pilot_events if e['event']=='started')
    pilot_end = max(datetime.fromisoformat(e['at_utc']) for e in pilot_events if e['event']=='finished')
    PILOT_TOTAL_SECONDS = (pilot_end-pilot_start).total_seconds()
    PROJECTED_TEST_SECONDS = PILOT_TOTAL_SECONDS / 20 * 762 * 1.25
    print('Pilot identity:', review['pilot_identity'])
    print('Pilot total seconds:', PILOT_TOTAL_SECONDS, 'Test planning minutes:', PROJECTED_TEST_SECONDS / 60)
    print('Budget projection within ceiling:', review['projected_total_ils'] <= 100)
    with (OUTPUT_ROOT / ('pilot-timing-' + str(time.time_ns()) + '.json')).open('x') as stream:
        json.dump({'pilot_total_seconds':PILOT_TOTAL_SECONDS,'projected_test_seconds':PROJECTED_TEST_SECONDS,'limit_seconds':5400},stream)
else:
    print('Pilot incomplete:', pilot_snapshot['n_completed'], '/20')


{"item_id": "ha-01004", "key": "5d3141e2e91b91b011ff3a49b9029b182dcd19bdfe96df3414bd6f80ae4ddfaa", "prompt": "בהתחשב במשפט הבא בעברית, מהו הפירוש של ראשי התיבות \"הרה״ר\"?\nענה במילים ספורות בלבד, ללא הסבר.\n\nמשפט: בראש ההנהגה הרוחנית של העדה והיישוב היהודי בארץ ישראל עמד הרה״ר, שכונה חכם באשי ובהמשך \"הראשון לציון\".\nפירוש:", "prompt_sha256": "37ffdac78467c16f3397daf36ee290ded3ba97ee03df24c0a614067626bfd040", "shown_order": null, "system": "gemini", "task": "generate", "status": "response_received", "response": "הרב הראשי", "selected_candidate": null, "correct": true, "valid": true, "attempts": 1, "retry_pending": false, "error": null, "response_metadata": {"attempts": 1, "candidate_count": 1, "error": null, "error_category": null, "finish_reason": "STOP", "http_status": 200, "identity_status": "verified", "identity_verification": "provider_reported_model_version", "model_version": "gemini-3.8-flash", "prompt_block_reason": null, "provider_blocked": false, "request_settings": {"gene

## 7. Test Evaluation and Resumption

Each qualified system is evaluated on all **381 test items in both tasks**, yielding **762 logical response records**. `INSPECTED_PILOTS` maps each inspected system name to the exact SHA-256 identity printed for its successful pilot. Before collection, the runner reloads that pilot, validates source, inputs, settings, and model identity, checks coverage, and recomputes the projection and shared expenditure.

Each additional system requires its own matching pilot. Unselected, uninspected, or blocked systems are reported explicitly. Resumption retains completed responses, failed attempts, ambiguous requests, and accumulated expenditure; completed responses are not requested again. All test identities remain in the cohort, including failed and unrun cases.

Freeze the source revision and settings during collection. A configuration change or repair of an explicit account block requires a new identified session and matching pilots, with previous artifacts preserved and expenditure carried forward. Deleting journals or changing identifiers to bypass a block is incompatible with this protocol.


In [14]:
INSPECTED_PILOTS = {'gemini': '6bddf30d08496d95b8c331bebfe72584be53c9aedba37fc610cd7a04f375f464'}
assert SELECTED_SYSTEMS == ['gemini']
assert PROJECTED_TEST_SECONDS <= 5400, 'Pilot planning estimate exceeds 90 minutes'
current_session()
FULL_CHUNKS = 40  # Each chunk remains limited to 20 new attempts.
for chunk in range(FULL_CHUNKS):
    test_summary = run_system_full(OUTPUT_ROOT, GEMINI_SYSTEM,
        inspected_pilot_identity=INSPECTED_PILOTS['gemini'], max_new_calls=20)
    terminal = test_summary['n_records'] - test_summary['n_pending'] - test_summary['n_ambiguous']
    events = saved_events(OUTPUT_ROOT / 'gemini/full-test')
    started = min(datetime.fromisoformat(e['at_utc']) for e in events if e['event'] == 'started')
    elapsed = (datetime.now(timezone.utc) - started).total_seconds()
    projection = elapsed / terminal * 762 * 1.25 if terminal else None
    checkpoint = {'n_calls':test_summary['n_calls'], 'n_completed':test_summary['n_completed'],
        'n_terminal':terminal, 'n_pending':test_summary['n_pending'],
        'n_ambiguous':test_summary['n_ambiguous'], 'elapsed_seconds':elapsed,
        'projected_minutes':projection / 60 if projection else None,
        'provider_states':test_summary.get('provider_states', {})}
    with (OUTPUT_ROOT / ('test-progress-' + str(time.time_ns()) + '.json')).open('x') as stream:
        json.dump(checkpoint, stream, indent=2)
    print(json.dumps(checkpoint), flush=True)
    if test_summary['n_ambiguous'] or test_summary.get('provider_states'):
        print('Paused: provider state or ambiguous request requires inspection.'); break
    if terminal >= 40 and projection > 5400:
        print('Paused: test planning estimate exceeds 90 minutes.'); break
    if not test_summary['n_pending']:
        print('All test requests reached a terminal state.'); break
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))


{"n_calls": 60, "n_completed": 60, "n_terminal": 60, "n_pending": 702, "n_ambiguous": 0, "elapsed_seconds": 174.778644, "projected_minutes": 46.24351622500001, "provider_states": {}}
{"n_calls": 80, "n_completed": 80, "n_terminal": 80, "n_pending": 682, "n_ambiguous": 0, "elapsed_seconds": 257.424581, "projected_minutes": 51.0826902921875, "provider_states": {}}
{"n_calls": 100, "n_completed": 100, "n_terminal": 100, "n_pending": 662, "n_ambiguous": 0, "elapsed_seconds": 299.420525, "projected_minutes": 47.53300834375, "provider_states": {}}
{"n_calls": 120, "n_completed": 120, "n_terminal": 120, "n_pending": 642, "n_ambiguous": 0, "elapsed_seconds": 345.488323, "projected_minutes": 45.70522606354166, "provider_states": {}}
{"n_calls": 140, "n_completed": 140, "n_terminal": 140, "n_pending": 622, "n_ambiguous": 0, "elapsed_seconds": 393.960948, "projected_minutes": 44.672357496428575, "provider_states": {}}
{"n_calls": 160, "n_completed": 160, "n_terminal": 160, "n_pending": 602, "n_am

## 8. Baselines and Encoder Limitations

The deterministic baseline evaluation includes uniform-random expectation, most-frequent expansion, most-mined expansion, and oracle selection. The trained DictaBERT candidate scorer and the untrained DictaBERT similarity baseline are separate systems; this notebook does not retrain either model.

The checkpoint adapter in `src/hebrew_acronyms/models/dictabert_cross_encoder/colab.py` supports the identified weights with SHA-256 `23bbff0324a7ce4d9d4a24a9ebfb87a4f6ac296bf1f1ce1bd25126c09260267f`. Checkpoint reuse also requires the corresponding model and tokenizer snapshot. A state dictionary alone does not establish the original training seed, selected epoch, library versions, or exact training examples. See `docs/checkpoints.md` in the source archive for the inference procedure and provenance limitations.

**Encoder evaluation limitation.** The historical test CSV lacks the explicit target spans required by the strict encoder interface. Multiple acronym occurrences cannot be resolved by silently selecting the first match. Fresh encoder evaluation requires qualified spans and is not performed by this notebook. Historical encoder results remain separately identified.


In [ ]:
from hebrew_acronyms.test_baselines import run_test_baselines
baseline_summary = run_test_baselines(
    TEST_PATH, REPO_DIR / "data/mined/candidate_table.csv",
    OUTPUT_ROOT / "baselines", code_revision=CODE_REVISION,
)
print(baseline_summary["summary"])

## 9. Result Artifacts and Interpretation

Each system's pilot and test directories contain a `manifest.json` identifying code, inputs, complete prompts, candidate orders, and settings; an `attempts.jsonl` journal retaining requests and outcomes; and a regenerated `summary.json` containing coverage and response records. `session.json` records the shared session configuration. Preserve the complete Drive directory and download an archive as a second copy.

Report results by system and task with the full cohort denominator, including missing or incomplete responses. Automatic selection-label parsing, string-based scores, and output-quality flags are distinct from human semantic judgments of generated expansions. Interpretation must retain source groups, returned model versions, run identities, and judgment provenance. Candidate labels beyond Z retain items with 30 candidates; comparisons with historical runs must account for this protocol difference.

`notebooks/experimental_study.ipynb` is the methods and analysis companion in the source archive. The existing development-set paper exporter is not a full-test exporter. Only explicitly identified run artifacts and evaluation procedures support reported results; downloading an archive does not itself validate the experiment or publish its findings.


In [15]:
# Preserve a verified session archive after collection stops.
from google.colab import files
final_summary = summarize_evaluation(OUTPUT_ROOT / 'gemini/full-test')
final_session_summary = session_summary(OUTPUT_ROOT)
print(json.dumps(final_summary['by_system_task'], ensure_ascii=False, indent=2))
print(json.dumps(final_session_summary, indent=2))
backup_stamp = str(time.time_ns())
receipt = {'run_name':RUN_NAME, 'code_revision':CODE_REVISION,
    'pilot_identity':INSPECTED_PILOTS['gemini'],
    'n_records':final_summary['n_records'], 'n_completed':final_summary['n_completed'],
    'n_pending':final_summary['n_pending'], 'n_ambiguous':final_summary['n_ambiguous'],
    'session_summary':final_session_summary,
    'files_sha256':{str(p.relative_to(OUTPUT_ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted(OUTPUT_ROOT.rglob('*')) if p.is_file()}}
with (OUTPUT_ROOT / ('collection-receipt-' + backup_stamp + '.json')).open('x') as stream:
    json.dump(receipt, stream, ensure_ascii=False, indent=2)
archive_path = Path(shutil.make_archive('/content/' + RUN_NAME + '-' + backup_stamp, 'zip', OUTPUT_ROOT))
archive_digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
backup_root = OUTPUT_ROOT.parent.parent / 'evaluation-backups'
backup_root.mkdir(parents=True, exist_ok=True)
drive_backup = backup_root / archive_path.name
with drive_backup.open('xb') as stream:
    stream.write(archive_path.read_bytes())
assert hashlib.sha256(drive_backup.read_bytes()).hexdigest() == archive_digest
print('VERIFIED_BACKUP', json.dumps({'archive':str(archive_path),'drive_backup':str(drive_backup),
    'sha256':archive_digest,'bytes':archive_path.stat().st_size}))
files.download(str(archive_path))


[
  {
    "system": "gemini",
    "task": "generate",
    "n_items": 381,
    "n_completed": 379,
    "status_counts": {
      "response_received": 379,
      "incomplete_response": 2
    },
    "accuracy": 0.5433070866141733,
    "score": "historical_quote_normalized_gold_substring",
    "n_valid": 214
  },
  {
    "system": "gemini",
    "task": "select",
    "n_items": 381,
    "n_completed": 381,
    "status_counts": {
      "response_received": 381
    },
    "accuracy": 0.937007874015748,
    "score": "strict_candidate_label_v1",
    "n_valid": 381
  }
]
{
  "systems": {
    "qwen": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "gemini": {
      "dev_pilot": {
        "n_calls": 20,
        "n_completed": 20,
        "n_records": 20,
        "n_ambiguous": 0,
    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>